# Chapter 3: Boolean Masking, Views vs Copies & Signal Filtering
*Python for Data Analysis — Applied Engineering Edition*

> **Navigation:** [<< Course Roadmap](./00_roadmap_and_orientation.ipynb) | [<< Previous: Chapter 02](./session_02_numpy_arrays.ipynb) | **Current: Chapter 03** | [Next: Chapter 04 >>](./session_04_pandas_series_dataframe.ipynb)

---

## The Reality of Field Sensors: Noise, Spikes, and Dropouts

In textbooks, sensor data arrives clean, continuous, and well-behaved. In real buildings, physical telemetry is messy:
- Transmission glitches produce negative voltages or impossible spikes (like a chilled water temperature of 950°C).
- Disconnected analog probes report sensor dropouts (often encoded as `-99.0` or `-999.0`).
- Power meters experience momentary electrical surges.

If you pass dirty telemetry directly into an energy auditing model, your predictions will be completely distorted.

In this chapter, we master how to clean and filter telemetry at hardware speeds using NumPy's **Boolean Indexing**, understand the critical distinction between **Views and Copies** to avoid silent memory corruption, and perform conditional signal replacement with **`np.where`** and **`np.clip`**.

### The Big Picture: Conditioning a Dirty Sensor Stream in 5 Lines
Run the cell below (`Shift + Enter`) to see a top-down signal conditioning pipeline that eliminates sensor dropouts and clamps physical extremes without writing a single `for` loop:

In [ ]:
import numpy as np

# 1. Simulated raw chilled water temperature stream (°C) with transmission glitches
np.random.seed(42)
clean_signal = np.random.normal(loc=7.0, scale=1.2, size=1000)
dirty_signal = clean_signal.copy()

# Inject realistic hardware glitches: dropouts (-99.0) and sensor spikes (120.0°C)
dirty_signal[np.random.choice(1000, 20, replace=False)] = -99.0
dirty_signal[np.random.choice(1000, 15, replace=False)] = 120.0

# 2. Vectorized Signal Conditioning in 2 steps:
# Step A: Identify valid physical operating bounds (4°C to 18°C)
valid_mask = (dirty_signal >= 4.0) & (dirty_signal <= 18.0)

# Step B: Replace invalid readings with the median of valid data using np.where
baseline_temp = np.median(dirty_signal[valid_mask])
conditioned_signal = np.where(valid_mask, dirty_signal, baseline_temp)

print(f"Total readings:       {len(dirty_signal)}")
print(f"Corrupted readings:   {(~valid_mask).sum()}")
print(f"Dirty range:          [{dirty_signal.min():.1f}°C, {dirty_signal.max():.1f}°C]")
print(f"Conditioned range:    [{conditioned_signal.min():.1f}°C, {conditioned_signal.max():.1f}°C]")


---

## 1. Boolean Indexing: Querying Arrays by Condition

When you evaluate a comparison operator (`>`, `<`, `==`, `!=`) against a NumPy array, NumPy does not return a single `True` or `False`. Instead, it evaluates the comparison element-by-element and returns a **Boolean mask array** of identical shape.

You can then pass this boolean mask back into square brackets `arr[mask]` to extract only the elements where the mask is `True`.

Let's see how boolean filtering works in practice:

### A Quick Mental Check: Temperature Thresholds
Examine the code below. We have a small array of zone temperatures in °C.

**Question:** What will `temps > 24.0` produce? What will `temps[temps > 24.0]` return?
* Formulate your mental prediction, then run the cell:

In [ ]:
temps = np.array([21.5, 25.1, 23.8, 26.4, 22.0])

# Mental Prediction:
# Mask:   [False, True, False, True, False]
# Result: [25.1, 26.4]
overheating_mask = temps > 24.0
overheating_readings = temps[overheating_mask]

print("Boolean Mask:", overheating_mask)
print("Filtered Values:", overheating_readings)


### Hands-On: Detecting Undercooled Zones
In building automation, ASHRAE standards flag undercooled zones if temperature drops below `20.0°C`.

In the code cell below:
1. Modify the condition to find undercooled temperatures (`temps < 22.0`).
2. Run the cell and verify that it isolates `[21.5, 22.0]`.

In [ ]:
# Modify the condition below to temps < 22.0:
undercooled_mask = temps < 22.0
undercooled_readings = temps[undercooled_mask]

print("Undercooled Mask:", undercooled_mask)
print("Undercooled Readings:", undercooled_readings)


### Hands-On: Combining Multiple Range Criteria
To combine multiple conditions in NumPy, you must use bitwise operators (`&` for AND, `|` for OR, `~` for NOT) and **wrap every condition in parentheses**.

Suppose comfort zone criteria is strictly between `21.0°C` and `24.0°C` inclusive.

Complete the blanks below (replace `___` with the correct operator or number):

In [ ]:
readings = np.array([19.5, 21.5, 22.8, 23.9, 25.4, 27.1])

# Fill in the blanks:
# comfort_mask = (readings >= 21.0) & (readings <= 24.0)
comfort_mask = (readings >= 21.0) & (readings <= 24.0)
comfort_readings = readings[comfort_mask]

print("Comfort Mask:    ", comfort_mask)
print("Comfort Readings:", comfort_readings)


### Guided Exercise: Motor Hazard Detection
Now apply boolean indexing to a telemetry stream of 50,000 fan speed readings (in RPM).

Normal fan operation is between `500 RPM` and `1800 RPM`. Any reading `< 100 RPM` represents a stalled motor, while readings `> 2200 RPM` indicate hazardous overspeed.

**Your Task:**
1. Create a boolean mask `hazard_mask` identifying readings where `fan_rpm < 100` OR `fan_rpm > 2200`.
2. Extract the hazardous readings into `hazard_readings = fan_rpm[hazard_mask]`.
3. Compute `hazard_count = len(hazard_readings)`.
4. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
np.random.seed(101)
fan_rpm = np.random.normal(loc=1200, scale=350, size=50_000)
# Inject hardware faults
fan_rpm[np.random.choice(50_000, 120, replace=False)] = 50.0   # stalls
fan_rpm[np.random.choice(50_000, 80, replace=False)] = 2450.0 # overspeed

# [YOUR CODE HERE]
hazard_mask = (fan_rpm < 100.0) | (fan_rpm > 2200.0)
hazard_readings = fan_rpm[hazard_mask]
hazard_count = len(hazard_readings)

# --- AUTOMATED TEST VERIFICATION ---
assert hazard_mask.shape == (50_000,), "hazard_mask shape mismatch"
assert hazard_mask.dtype == bool, "hazard_mask must be boolean"
assert (hazard_readings < 100.0).all() or (hazard_readings > 2200.0).all() or len(hazard_readings) > 0
assert hazard_count >= 200, f"Expected at least 200 hazard events, found {hazard_count}"
print(f"[PASS] All checks passed! Successfully isolated {hazard_count:,} hazardous motor events.")


---

## 2. Views vs. Copies: The Most Dangerous Pitfall in NumPy

This is the single most common cause of silent production bugs in scientific Python.

When you create a slice of an array (e.g., `sub_arr = arr[0:5]`):
* NumPy **DOES NOT** copy the numbers into new memory.
* It creates a **View**: a new metadata header pointing directly to the **SAME underlying memory buffer**!
* **Danger:** If you modify `sub_arr`, you silently mutate the original `arr`!

Conversely, **Boolean Indexing (`arr[mask]`) and Fancy Indexing always create a COPY**.

Let's test this distinction directly in code:

### A Quick Mental Check: Mutating a Slice
Examine the code below. We slice the first 3 elements of `original_grid`, and set the first element of `slice_view` to `999.0`.

**Question:** Will `original_grid[0]` remain `10.0` or become `999.0`?
* Formulate your mental prediction, then run the cell:

In [ ]:
original_grid = np.array([10.0, 20.0, 30.0, 40.0, 50.0])

# Slicing creates a VIEW
slice_view = original_grid[:3]
slice_view[0] = 999.0

print("slice_view:    ", slice_view)
print("original_grid: ", original_grid)  # original_grid[0] was mutated!
print("Shares memory? ", np.shares_memory(slice_view, original_grid))


### Hands-On: Safeguarding Data with .copy()
If you want an independent array that does not modify the original data buffer, you must explicitly call **`.copy()`**.

In the code cell below:
1. Add `.copy()` to the slice assignment: `safe_copy = original_temps[:3].copy()`.
2. Run the cell and verify that `original_temps` is NOT mutated.

In [ ]:
original_temps = np.array([18.0, 19.5, 21.0, 22.5])

# Modify the line below by adding .copy():
safe_copy = original_temps[:2].copy()
safe_copy[0] = -50.0

print("safe_copy:      ", safe_copy)
print("original_temps: ", original_temps)  # Untouched!
print("Shares memory?  ", np.shares_memory(safe_copy, original_temps))


### Hands-On: Boolean Subsets and Shared Memory
Remember: **Boolean indexing always returns a copy**, not a view.
However, modifying an array through boolean indexing directly on the LHS (`arr[mask] = val`) modifies `arr` in place.

Complete the blanks below (replace `___` with the correct method or expression):

In [ ]:
power_readings = np.array([45.2, -99.0, 52.1, -99.0, 48.7])

# Fill in the blanks:
# Extract valid values into an independent array:
valid_mask = power_readings > 0
valid_subset = power_readings[valid_mask]

# Check if valid_subset shares memory with power_readings (False):
shares = np.shares_memory(valid_subset, power_readings)
print("Valid subset:   ", valid_subset)
print("Shares memory?  ", shares)


### Guided Exercise: Protecting Master Calibration Buffers
Now verify your understanding of memory safety on a baseline dataset.

**Your Task:**
1. Given `master_telemetry`, extract the first 50 rows into `calibration_window` as a SAFE COPY (using `.copy()`).
2. Set all values in `calibration_window` to `0.0`.
3. Verify that `master_telemetry` was NOT changed.
4. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
np.random.seed(42)
master_telemetry = np.random.uniform(20.0, 80.0, size=(100, 3))
original_first_val = master_telemetry[0, 0]

# [YOUR CODE HERE]
calibration_window = master_telemetry[:50].copy()
calibration_window[:] = 0.0

# --- AUTOMATED TEST VERIFICATION ---
assert not np.shares_memory(calibration_window, master_telemetry), "calibration_window must NOT share memory with master_telemetry"
assert master_telemetry[0, 0] == original_first_val, "master_telemetry was accidentally mutated!"
assert (calibration_window == 0.0).all(), "calibration_window was not set to 0.0"
print("[PASS] All checks passed! Memory safety successfully preserved.")


---

## 3. Signal Conditioning with `np.where` and `np.clip`

Two NumPy functions form the backbone of signal processing:
1. **`np.where(condition, x, y)`**: Vectorized ternary operator (`condition ? x : y`). Where condition is True, yield `x`; else yield `y`.
2. **`np.clip(arr, a_min, a_max)`**: Fast clamping of all values outside `[a_min, a_max]` to the boundary limits.

Let's see how both functions operate on sensor data:

### A Quick Mental Check: Conditional Replacement
Examine the code below. We have battery state-of-charge voltages.
Any reading `< 0.0` is an error code that should be replaced with `0.0`. Valid voltages should remain untouched.

**Question:** What will `np.where(raw_v >= 0.0, raw_v, 0.0)` produce?
* Formulate your mental prediction, then run the cell:

In [ ]:
raw_v = np.array([3.7, -99.0, 4.1, 3.9, -1.0])

# Mental Prediction: [3.7, 0.0, 4.1, 3.9, 0.0]
cleaned_v = np.where(raw_v >= 0.0, raw_v, 0.0)
print("Raw voltages:    ", raw_v)
print("Cleaned voltages:", cleaned_v)


### Hands-On: Fallback Baseline Replacement
Instead of replacing invalid numbers with `0.0`, in physical telemetry we often replace invalid values with a **fallback default baseline** (such as `22.0°C`).

In the code cell below:
1. Modify `np.where` so that values `< 0.0` are replaced with `default_temp = 22.0`.
2. Run the cell and verify the result.

In [ ]:
probe_temps = np.array([21.4, -999.0, 23.1, -999.0, 22.8])
default_temp = 22.0

# Modify the fallback value in np.where below:
repaired_temps = np.where(probe_temps >= 0.0, probe_temps, default_temp)
print("Repaired Temperatures:", repaired_temps)


### Hands-On: Clamping Physical Bounds with np.clip
When you know physical sensors cannot exceed an operational band (e.g., HVAC supply air must stay between `10.0°C` and `30.0°C`), **`np.clip`** clamps spikes to the exact boundary limits in one C call.

Complete the blanks below (replace `___` with the correct function and parameters):

In [ ]:
air_temps = np.array([8.5, 12.0, 24.5, 32.8, 19.0])

# Fill in the blanks to clamp air_temps between 10.0 and 30.0:
# clamped_temps = np.clip(air_temps, 10.0, 30.0)
clamped_temps = np.clip(air_temps, 10.0, 30.0)

print("Original: ", air_temps)
print("Clamped:  ", clamped_temps)


### Guided Exercise: Cleaning Telemetry Surges and Dropouts
Now apply both `np.where` and `np.clip` to a simulated building electrical power stream.

**Your Task:**
1. Disconnected sensors report negative dropout values (`-99.0`). Replace all negative readings in `raw_kw` with the **median of the positive readings** using `np.where` to create `positive_kw`.
2. Then, clamp `positive_kw` between `0.0 kW` and `250.0 kW` (the main circuit breaker limit) using `np.clip` to create `final_kw`.
3. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
np.random.seed(42)
raw_kw = np.random.normal(loc=110.0, scale=35.0, size=10_000)
# Inject dropouts and lightning surges
raw_kw[np.random.choice(10_000, 50, replace=False)] = -99.0
raw_kw[np.random.choice(10_000, 20, replace=False)] = 850.0

# [YOUR CODE HERE]
valid_median = np.median(raw_kw[raw_kw >= 0.0])
positive_kw = np.where(raw_kw >= 0.0, raw_kw, valid_median)
final_kw = np.clip(positive_kw, 0.0, 250.0)

# --- AUTOMATED TEST VERIFICATION ---
assert (final_kw >= 0.0).all(), "final_kw still contains negative values!"
assert (final_kw <= 250.0).all(), "final_kw contains values above 250 kW breaker limit!"
assert np.isclose(final_kw.max(), 250.0), "Max value should equal upper clip limit 250.0"
print(f"[PASS] All checks passed! Stream conditioned cleanly. Final range: [{final_kw.min():.1f}, {final_kw.max():.1f}] kW")


---

## 4. Learning Through Deliberate Mistakes

Let's deliberately trigger two of the most infamous bugs in Python data science:

### Trap 1: The Truth Value of an Array is Ambiguous
What happens when you write `if (arr > 5) and (arr < 10):` using Python's standard `and` keyword?

Standard Python `and` tries to evaluate the entire array as a single boolean (`bool(arr)`). Because an array has multiple elements, Python cannot decide whether the array is collectively True or False!

In [ ]:
arr = np.array([2, 6, 8, 12])

# # If you uncomment the line below, it triggers:
# # ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
# bad_mask = (arr > 5) and (arr < 10)

# THE FIX: Always use bitwise & with parentheses for element-wise boolean operations:
good_mask = (arr > 5) & (arr < 10)
print("Bitwise mask resolved cleanly! Mask:", good_mask)
print("Filtered elements:", arr[good_mask])


### Trap 2: Slicing Without Copying Causes Spooky Action at a Distance
Imagine you normalize a baseline time-slice of your sensor data, only to find that your raw historical archive was silently overwritten!

In [ ]:
telemetry_history = np.array([100.0, 200.0, 300.0, 400.0])

# BUG: Slicing directly creates a view
baseline_window = telemetry_history[:2]
baseline_window /= 10.0  # In-place division mutates telemetry_history!

print("Bugged history (corrupted!):", telemetry_history)

# THE FIX: Always create an explicit copy when transforming a subset:
clean_history = np.array([100.0, 200.0, 300.0, 400.0])
safe_window = clean_history[:2].copy()
safe_window /= 10.0
print("Safe history (protected!):  ", clean_history)


---

## 5. Real-World Challenge: Cleaning a 3-Site Building Fleet

Now put everything together in a complete engineering pipeline on real telemetry data from our `data/` folder!

Run the cell below to load our sample Building Data Genome 2 telemetry dataset:

In [ ]:
import pandas as pd
import numpy as np

# Load 87,600 hourly telemetry observations from our sample BDG2 dataset
df_telemetry = pd.read_parquet("data/building_telemetry.parquet")
print(f"Loaded {len(df_telemetry):,} records for {df_telemetry['building_id'].nunique()} buildings.")

# Extract raw electrical power array for 'Office_Alicia'
alicia_kw = df_telemetry[df_telemetry["building_id"] == "Office_Alicia"]["reading_kw"].to_numpy()
print(f"Office_Alicia readings count: {len(alicia_kw)} | dtype: {alicia_kw.dtype}")


### Your Engineering Challenge
In the code cell below, complete the following analysis for `Office_Alicia`:
1. **Analyze Peak vs Baselline:**
   - Define a peak load threshold: `peak_threshold = np.percentile(alicia_kw, 90)` (top 10% highest power hours).
   - Create a boolean mask `peak_mask` identifying readings `>= peak_threshold`.
   - Compute the average power during peak hours: `mean_peak = alicia_kw[peak_mask].mean()`.
   - Compute the average power during non-peak hours: `mean_offpeak = alicia_kw[~peak_mask].mean()`.
2. **Simulate Demand Response Curtailment:**
   - Create a safe copy `curtailed_kw = alicia_kw.copy()`.
   - Using `np.where`, during peak hours reduce power demand by 20% (`curtailed_kw * 0.80`), while keeping off-peak hours unchanged.
3. **Compute Energy Savings:**
   - Total annual kWh before curtailment: `total_kwh_before = alicia_kw.sum()`.
   - Total annual kWh after curtailment: `total_kwh_after = curtailed_kw.sum()`.
   - Savings percentage: `savings_pct = (1.0 - total_kwh_after / total_kwh_before) * 100.0`.
4. Run the assert test cell below to verify your solution!

In [ ]:
# [YOUR CODE HERE]
# Task 1: Peak vs Off-peak analysis
peak_threshold = np.percentile(alicia_kw, 90)
peak_mask = alicia_kw >= peak_threshold
mean_peak = alicia_kw[peak_mask].mean()
mean_offpeak = alicia_kw[~peak_mask].mean()

# Task 2: Demand Response Curtailment with np.where
curtailed_kw = np.where(peak_mask, alicia_kw * 0.80, alicia_kw)

# Task 3: Energy Savings Calculation
total_kwh_before = alicia_kw.sum()
total_kwh_after = curtailed_kw.sum()
savings_pct = (1.0 - total_kwh_after / total_kwh_before) * 100.0

print(f"Peak Threshold:   {peak_threshold:.2f} kW")
print(f"Mean Peak:        {mean_peak:.2f} kW | Mean Off-Peak: {mean_offpeak:.2f} kW")
print(f"Annual Savings:   {savings_pct:.2f}%")


In [ ]:
# --- AUTOMATED TEST VERIFICATION ---
assert peak_mask.sum() == int(len(alicia_kw) * 0.10), "peak_mask should capture exactly the top 10% of hours"
assert mean_peak > mean_offpeak, "Mean peak power must be higher than off-peak power"
assert not np.shares_memory(curtailed_kw, alicia_kw), "curtailed_kw must not share memory with alicia_kw"
assert savings_pct > 0.0 and savings_pct < 10.0, f"Expected realistic curtailment savings (3-8%), got {savings_pct:.2f}%"
print("[SUCCESS] Congratulations! All demand-response curtailment assertions passed successfully!")


---

## 6. Key Takeaways & Review Routine

Here are the core takeaways to master from Chapter 3:

1. **Boolean Masking** evaluates comparisons element-by-element; use bitwise `&`, `|`, and `~` with parentheses for compound filtering.
2. **Views vs Copies**: Slicing creates a *view* (mutations modify the parent array). Boolean indexing and explicit `.copy()` create independent *copies*.
3. **Signal Conditioning**: Use `np.where(cond, x, y)` for conditional value replacement and `np.clip(arr, min, max)` for fast hardware boundary clamping.

### Quick Retrieval Check (Task ➔ Code)
Test your active recall before moving forward:
* **Prompt 1:** How do you filter an array `arr` to include only values between 10 and 50 inclusive?
  * *Answer:* `arr[(arr >= 10) & (arr <= 50)]`
* **Prompt 2:** How do you test if two arrays `a` and `b` share the same underlying RAM data buffer?
  * *Answer:* `np.shares_memory(a, b)`
* **Prompt 3:** How do you replace all negative numbers in `signals` with `0.0` while keeping positive numbers untouched?
  * *Answer:* `np.where(signals >= 0, signals, 0.0)`

### Coming Up in Chapter 4:
In our next session, we transition from raw NumPy arrays to **Pandas Series & DataFrames**: structured labeled telemetry, handling mixed column data types, and index-aligned operations.

👉 **Next Step:** Open **[Chapter 04: Labeled Telemetry with Pandas Series & DataFrames](./session_04_pandas_series_dataframe.ipynb)** to continue.